## Business Problem

The objective is to develop a predictive maintenance system that can classify engine condition based on sensor data such as RPM, oil pressure, fuel pressure, and temperature readings.

This will help:
- Reduce unexpected engine failures
- Optimize maintenance schedules
- Improve operational efficiency

# Imports

In [ ]:
import os
import sys


from datasets import load_dataset
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from dotenv import load_dotenv
from huggingface_hub import HfApi
from sklearn.model_selection import train_test_split

import joblib

## Load Data (From Hugging Face)

In [ ]:
load_dotenv()
repo_name = os.getenv("HF_REPO_DATA")

dataset = load_dataset(repo_name)

df = dataset['train'].to_pandas()

df.head()

## Basic Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Observations:
- Dataset contains 19535 rows and 7 features
- No missing values detected
- Features include Engine rpm, Lub oil pressure, Fuel pressure, Coolant pressure,lub oil temp,Coolant temp	metrics

## Missing Values

In [ ]:
df.isnull().sum()

There are no missing values in the dataset

## Univariate Analysis

In [ ]:

for column in df.columns:
    df[column].hist(figsize=(4,3))
    plt.title(f"{column} distribution")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.show()


📊 Univariate Analysis - Engine Sensor Feature

🔍 **Distribution Overview**
🚗 Engine RPM
* Distribution seems to be Right-Skewed
* Most values concentrated between 500 to 1000 rpm
* Few High RPM outliers (> 1500)

👉 **Interpretation**
* Engine typically operates in stable RPM
* High RPM may indicate some heavy load on engine

💧 **Lubrication**
* Almost normal distribution
* Centered around 3-4 unit
* There are extreme high and low values

👉 **Interpretation:**
* Oil pressure is relative stable
* Extreme deviations may indicate lubrication issue

⛽ **Fuel Pressure**
* Slight right-skewed
* Most value between 5-10
* Some high value outliers

👉 **Interpretation:**
* Fuel delivery is mostly stable
* High pressure spikes may impact combustion efficiency

🌡️ **Coolant Pressure**
* Concentrated around 1–3
* Long tail toward higher values

👉 **Interpretation:**

* Normal cooling system operates in a narrow band
* High pressure could indicate cooling system stress or blockage

🔥 **Lubrication Oil Temperature**
* Narrow distribution (~75–85°C)
* Slight right skew

👉 **Interpretation:**

* Oil temperature is tightly controlled
* Elevated values may reduce lubrication efficiency

❄️ **Coolant Temperature**
* Concentrated around 70–95°C
* Some higher-end spread

👉 **Interpretation:**

* Normal operating range visible
* Higher temperatures suggest overheating risk

⚙️ **Engine Condition (Target Variable)**
* Binary (0 and 1)
* Appears reasonably balanced

👉 **Interpretation:**

* No severe class imbalance → good for modeling
* No immediate need for resampling techniques

### Key Insights

### Key Observations from Univariate Analysis

1. Most engine parameters operate within well-defined ranges, indicating stable system behavior under normal conditions.

2. Temperature-related features (coolant and oil temperature) show tighter distributions, suggesting controlled operating environments.

3. Pressure-related features exhibit slight skewness and outliers, which may correspond to abnormal or failure conditions.

4. Engine RPM shows variability with some extreme values, potentially indicating high load or stress scenarios.

5. The target variable (Engine Condition) is relatively balanced, enabling effective supervised learning without additional resampling.

### Implications for Modeling

- Features with narrow distributions (temperature) may require careful scaling
- Outliers should be analyzed, not blindly removed, as they may represent real failure events
- No class imbalance simplifies model training and evaluation

## Target Variable Analysis

In [ ]:
sns.countplot(x='Engine Condition', data=df)
plt.show()

## Bivariate Analysis

In [ ]:
features = df.columns.drop("Engine Condition")

for col in features:
    sns.boxplot(x='Engine Condition', y=col, data=df)
    plt.title(col)
    plt.show()

📊 Bivariate Analysis — Engine Features vs Engine Condition

🔹 **Engine RPM vs Engine Condition**

* Median RPM is higher for normal engines (0) than faulty engines (1)
* Faulty engines tend to operate at lower RPM
* Significant overlap exists between both classes
* Outliers are present in both categories

👉**Interpretation**

Engine degradation is associated with lower RPM, but RPM alone cannot clearly distinguish engine condition due to overlap.

Conclusion

Engine RPM is a weak standalone predictor and should be combined with other features.

🔹 Lubrication Oil Pressure vs Engine Condition

From the boxplot, we observe:

Slight increase in median oil pressure for faulty engines
Distributions of both classes are very similar
Presence of low-value outliers in both conditions
Interpretation

Lubrication oil pressure does not show strong variation between normal and faulty engines, indicating limited predictive power.

Conclusion

Oil pressure alone is not a strong indicator of engine failure but may contribute when combined with other features.

🔹 Fuel Pressure vs Engine Condition

From the boxplot, we observe:

Faulty engines (1) show slightly higher median fuel pressure
Wider spread and more high-value outliers in faulty engines
Some separation compared to previous features
Interpretation

Higher fuel pressure may be associated with faulty engines, possibly due to irregular fuel supply or combustion issues.

Conclusion

Fuel pressure has moderate predictive importance and can help distinguish engine condition when used with other variables.

🔹 Coolant Pressure vs Engine Condition

From the boxplot, we observe:

Almost identical distribution for both engine conditions
No significant difference in median values
Outliers present in both classes
Interpretation

Coolant pressure does not vary significantly between normal and faulty engines.

Conclusion

Coolant pressure is a weak predictor and may have minimal impact on model performance.

🔹 Lubrication Oil Temperature vs Engine Condition

From the boxplot, we observe:

Very similar distributions for both classes
Slightly higher variability in faulty engines
Presence of high-temperature outliers
Interpretation

Oil temperature remains relatively stable across engine conditions, with only minor variation.

Conclusion

Lubrication oil temperature alone is not a strong predictor but may support other features.

🔹 Coolant Temperature vs Engine Condition

From the boxplot, we observe:

Slight increase in temperature for faulty engines
Presence of extreme high-temperature outliers in faulty engines
Moderate overlap between classes
Interpretation

Higher coolant temperatures may indicate overheating, which is associated with engine faults.

Conclusion

Coolant temperature is an important feature and can act as an early indicator of engine failure.

## Final Bivariate Summary

### Overall Observations

- Temperature-related features (especially coolant temperature) show better separation between normal and faulty engines
- Fuel pressure shows moderate predictive capability
- RPM shows some variation but has significant overlap
- Oil pressure and coolant pressure have minimal impact individually

### Key Takeaway

No single feature can fully predict engine condition. A combination of temperature, pressure, and RPM features will be required to build an effective predictive model.

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

📊 Multivariate Analysis — Correlation Heatmap

From the correlation heatmap, we observe the relationships between different engine parameters and the target variable.

🔍 Observations

- Most features have low correlation with each other, indicating minimal multicollinearity
- The highest correlation with the target variable (Engine Condition) is:
  - Engine RPM (~ -0.27) → moderate negative correlation
  - Fuel Pressure (~ 0.12) → weak positive correlation
- Other features such as:
 - Lubrication Oil Pressure
 - Coolant Pressure
 - Temperatures
show very weak correlation with the target

- Temperature features (Lub Oil Temp & Coolant Temp) show slight correlation with each other

🧠 Interpretation

- Engine RPM has the strongest relationship with engine condition, where lower RPM is associated with faulty engines
- Fuel pressure shows a small positive relationship with failure
- Most features individually have weak correlations, indicating that no single feature can predict engine failure effectively

💼 Business Insight
- Engine failure is not driven by a single parameter but by a combination of multiple factors
- RPM and fuel pressure provide some signal, but temperature and pressure interactions are likely more important
- The weak correlations suggest the need for machine learning models that can capture non-linear relationships

📌 Key Takeaways
- No strong multicollinearity → good for model stability
- No dominant single feature → need for ensemble models
- Feature interactions will play a key role in prediction

✅ Conclusion

Correlation analysis shows that individual features have limited predictive power. A combination of multiple parameters using advanced machine learning models will be required to accurately predict engine condition.

## Final EDA Summary

- RPM shows moderate correlation with engine condition
- Fuel pressure has some predictive signal
- Temperature and pressure features individually show weak relationships
- No strong multicollinearity observed among features

Overall, engine failure prediction requires combining multiple features rather than relying on any single parameter.

## Outlier Detection

In [ ]:
for col in df.columns:
    sns.boxplot(df[col])
    plt.title(col)
    plt.show()

📊 Outlier Detection — Engine Features


🔍 Observations

From the boxplots, we can observe the presence of outliers across multiple features:

🚗 **Engine RPM**

- Significant number of high-value outliers (>1500)
- Some very low values also present

💧 **Lubrication Oil Pressure**

- Outliers on both lower and higher ends
- Few extremely low values close to zero

⛽ **Fuel Pressure**

-Large number of high-value outliers
-Wide spread compared to other features

🌡️ **Coolant Pressure**

- Moderate number of high-value outliers
- Distribution mostly stable otherwise

🔥 **Lubrication Oil Temperature**

- Several high-temperature outliers
- Overall distribution remains tight

❄️ **Coolant Temperature**

- Few extreme outliers (very high values ~120–200)
- Majority of values lie in a stable range


👉 **Interpretation**
- Outliers are present in almost all features, especially in RPM and pressure-related variables
- These outliers likely represent extreme operating conditions or potential failure scenarios, rather than noise
- Temperature features show fewer but more critical outliers (possible overheating cases)

💼 ** Business Insight**

- High RPM and pressure spikes may indicate engine stress or overload conditions
- Extremely high coolant temperature is a strong signal of overheating and possible failure
- Very low oil pressure values may indicate lubrication failure, which is critical for engine health


⚠️ **Important Consideration**

Outliers should not be removed without doing further analysis, as they may represent real-world failure cases that are important for predictive maintenance.

✅ Conclusion

Outliers are present across multiple features and are likely required to be there. Instead of removing them,we should handle them during preprocessing phase to ensure the model captures critical failure patterns.

> Outliers will be retained during preprocessing, as they represent important edge cases relevant to engine failure prediction.

## Preprocessing

There are no duplicates found, no missing values and no type conversion issue so not performing any preprocessing step for data cleansing

In [ ]:
# Prepare data
X = df.drop("Engine Condition", axis=1)
y = df["Engine Condition"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
#Saving dataset locally

os.makedirs("data/processed", exist_ok=True)

train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
train.to_csv("data/processed/train.csv", index=False)
test.to_csv("data/processed/test.csv", index=False)

In [ ]:
def upload_data_to_hf():

    load_dotenv()
    repo_name = os.getenv('HF_REPO_DATA')
    if not repo_name:
        raise ValueError("REPO_NAME environment variable not set.")
    
    api = HfApi()

    # Upload the processed data to Hugging Face Hub
    api.upload_folder(
        folder_path="data/processed",
        repo_id=repo_name,
        repo_type="dataset",
        path_in_repo="processed"
    )

    print("Data uploaded successfully to Hugging Face Hub.")

In [ ]:
# Uploading the train and test datasets to Hugging Face Hub
upload_data_to_hf()

The integration with Hugging Face Hub enables version control, reproducibility, and centralized model management, aligning with industry-standard MLOps practices.

## Model Training and Evaluation

The following section compares multiple machine learning models and selects the best model based on F1-score.

In [ ]:

# Random Forest
rf = RandomForestClassifier(random_state=79)
rf.fit(X_train, y_train)


In [ ]:
rf_preds = rf.predict(X_test)

In [ ]:

# XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)


In [ ]:
xgb_preds = xgb.predict(X_test)

## Evaluate Models

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }



In [ ]:
rf_metrics = evaluate(y_test, rf_preds)
xgb_metrics = evaluate(y_test, xgb_preds)



In [ ]:
#Printing metrics
def print_metrics(metrics_dict, model_name):
    print(f"Metrics for {model_name}:")
    for metric, value in metrics_dict.items():
        print(f"{metric}: {value:.4f}")
    print("\n") 

In [ ]:
print_metrics(rf_metrics, "Random Forest")
print_metrics(xgb_metrics, "XGBoost")

## Best Model Selection

The final model was selected based on the F1-score, as it provides a balanced measure of both precision and recall. This is particularly important in predictive maintenance scenarios where both false positives and false negatives have significant operational impact. The selected model demonstrated the best trade-off between these metrics, indicating strong generalization capability.

So the best model is `Random Forest`

## Hyperparameter Tuning

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="f1"
)

grid.fit(X_train, y_train)


In [ ]:

best_model = grid.best_estimator_
print("Best Params:", grid.best_params_)

## Final Evaluation

In [ ]:
final_preds = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, final_preds))
print("Final Precision:", precision_score(y_test, final_preds))
print("Final Recall:", recall_score(y_test, final_preds))
print("Final F1:", f1_score(y_test, final_preds))

The tuned model achieves comparable performance to the baseline, indicating stable generalization without overfitting.

## Save Model

In [ ]:

os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/engine_maintenance_model.pkl")

## Upload Dataset to HuggingFace


Creating a common function to upload the model to Hugging Face

In [ ]:
# Uploading the model to Hugging Face Hub
def upload_model_to_hf(model_path="model/engine_maintenance_model.pkl"):
    load_dotenv()

    # get token and repo name from environment variables
    token = os.getenv("HF_TOKEN")
    repo_name = os.getenv("HF_REPO_MODEL")

    api = HfApi()

    # Create repo if not exists
    api.create_repo(
        repo_id=repo_name,
        repo_type="model",
        exist_ok=True,
        token=token
    )

    # Upload model file
    api.upload_file(
       path_or_fileobj=model_path,
        path_in_repo=os.path.basename(model_path),
        repo_id=repo_name,
        token=token
    )

    print(f"Model uploaded to Hugging Face: {repo_name}")

# Upload Model to Hugging Face

In [ ]:
# upload the model to Hugging Face Hub
upload_model_to_hf()

This notebook demonstrates the end-to-end MLOps workflow including data loading, EDA, preprocessing, model training, evaluation, and deployment.